# 🔀 Multi-Agent Research and Summarization: Router, RAG, Web Search and Summarizer

A research assistant that looks at each question and sends it to the right specialist: the restaurant's own
**knowledge base** (RAG), a **web search**, or the **LLM** on its own. Whatever the specialist returns, a
**summarization agent** turns it into the final structured answer. The knowledge base describes *Food Factory*,
a fictional fine-dining restaurant in Bangalore.

## Learning Objectives
In this notebook, you will learn:
1. **A RAG agent** - Index a knowledge base in Chroma and answer from the retrieved chunks only
2. **A web research agent** - Search DuckDuckGo (`ddgs`) and extract the relevant facts with their sources
3. **A typed router** - Structured output with a `Literal` route, and a *pure* routing function that reads it
4. **A summarization agent** - Synthesize the one specialist result this question produced into a final answer
5. **Follow-up questions** - A checkpointer plus a small `history` list, so "what about 40 people?" makes sense

## Where this fits
`12_supervisor_financial_research.ipynb` had a supervisor delegate to a team, turn after turn. Here a router
makes **one** decision per question and exactly one specialist runs. `14_multi_agent_research_system.ipynb`
then runs several workers *in parallel* and loops on the result's quality.

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- Internet access for the DuckDuckGo search (no API key needed)
- `restaurant_knowledge_base.py` next to this notebook: the Food Factory knowledge base as a Python module

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks, including the knowledge base module.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import json
import operator
from datetime import date
from typing import Annotated, Literal, TypedDict

from aied_helpers import get_embeddings
from ddgs import DDGS
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

# Companion module in this folder: the restaurant's knowledge base as plain Python data
from restaurant_knowledge_base import FOOD_FACTORY_KNOWLEDGE_BASE, MENU_DATA, RESERVATION_INFO
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the router, the specialists and the summarizer
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📚 Part 1: The Knowledge Base

`restaurant_knowledge_base.py` holds three things: a long text about the restaurant (menu, prices, events,
policies, FAQ) and two dictionaries (`MENU_DATA`, `RESERVATION_INFO`). We join them into one text, split it into
~1,000-character chunks at the `===` section headings where possible, and index the chunks in an **in-memory**
Chroma collection with OpenAI `text-embedding-3-small` embeddings and cosine distance.

`reset_collection()` empties the collection first, so re-running the cell never duplicates chunks.

In [ ]:
# ============================================================================
# SPLIT AND INDEX THE KNOWLEDGE BASE
# ============================================================================
full_content = (
    FOOD_FACTORY_KNOWLEDGE_BASE
    + "\n\n=== MENU DATA (STRUCTURED) ===\n" + json.dumps(MENU_DATA, indent=2, ensure_ascii=False)
    + "\n\n=== RESERVATION INFORMATION (STRUCTURED) ===\n" + json.dumps(RESERVATION_INFO, indent=2, ensure_ascii=False)
)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150, separators=["\n\n===", "\n\n", "\n", " ", ""]
)
chunks = splitter.split_text(full_content)

vectorstore = Chroma(
    collection_name="food_factory_kb",
    embedding_function=get_embeddings(provider="openai", model="text-embedding-3-small"),
    collection_configuration={"hnsw": {"space": "cosine"}},  # cosine distance (Chroma's default is L2)
)
vectorstore.reset_collection()
vectorstore.add_texts(chunks)

print(f"✅ {len(chunks)} chunks from {len(full_content):,} characters")
print(f"   Chunks stored in the collection: {len(vectorstore.get()['ids'])}")

### 1.1 🔎 A retrieval helper

`retrieve_restaurant()` returns the top `k` chunks **with their relevance scores** and drops weak matches
(score below `min_score`), so a question the knowledge base cannot answer returns nothing instead of three
unrelated chunks.

In [ ]:
# ============================================================================
# RETRIEVE WITH SCORES
# ============================================================================
def retrieve_restaurant(query: str, k: int = 3, min_score: float = 0.2) -> list[tuple]:
    """Top-k (chunk, relevance score) pairs above min_score (1.0 = identical meaning)."""
    hits = vectorstore.similarity_search_with_relevance_scores(query, k=k)
    return [(doc, score) for doc, score in hits if score >= min_score]


def chunk_label(text: str) -> str:
    """First non-empty line of a chunk, as a short source label."""
    first = next((line.strip(" =*") for line in text.splitlines() if line.strip(" =*")), "")
    return first[:70]


question = "Can I host a private event for 20 people?"
hits = retrieve_restaurant(question)
print(f"🔎 {question}\n")
for doc, score in hits:
    print(f"  {score:.2f}  {chunk_label(doc.page_content)}")

found = any("8-24" in doc.page_content for doc, _ in hits)
print(f"\n{'✅' if found else '❌'} a retrieved chunk states the private dining room's 8-24 guest capacity")

**Reading the output:** each line is one retrieved chunk with its relevance score. The ✅ line checks the
retrieved text itself, so the RAG agent below has the capacity fact available to answer from.

---
## 🧠 Part 2: The Shared State

Every node reads from and writes to one `AgentState`. Two design choices matter here:

- **Nodes return partial updates.** A node returns only the keys it changed; it never mutates `state`.
- **Per-question fields are reset by the router.** With a checkpointer the state survives between questions on
  a thread, so a web result from question 1 would otherwise still be there when question 2 is summarized.

### Key Concepts:
- **`standalone_query`**: the router's rewrite of the question so it makes sense without the conversation
  (the specialists search with this, not with the raw follow-up)
- **`history: Annotated[list[dict], operator.add]`**: a reducer; each answered question is *appended*, which is
  how a follow-up question sees what came before

In [ ]:
# ============================================================================
# STATE DEFINITION
# ============================================================================
class AgentState(TypedDict):
    query: str               # the question as the user typed it
    standalone_query: str    # the router's self-contained rewrite of it
    route_decision: str      # "web_research" | "rag" | "llm"
    route_reason: str
    web_results: str
    rag_results: str
    llm_response: str
    sources: list[str]       # chunk labels (rag) or URLs (web) behind this question's answer
    final_summary: str
    history: Annotated[list[dict], operator.add]  # {"query", "answer"} per answered question


print("✅ AgentState fields:", ", ".join(AgentState.__annotations__))

---
## 🍽️ Part 3: The RAG Agent

The RAG agent retrieves chunks for the standalone question and asks the model to answer **from those chunks
only**. If nothing relevant is retrieved it does not call the model at all and says so, pointing to the
restaurant's contact details.

Nodes are plain functions of the state, so we can call one directly to test it before wiring the graph.

In [ ]:
# ============================================================================
# RAG AGENT, TESTED ON ITS OWN
# ============================================================================
def restaurant_rag_agent(state: AgentState) -> dict:
    query = state.get("standalone_query") or state["query"]
    hits = retrieve_restaurant(query)
    if not hits:
        print("⚠️ RAG: no relevant chunks")
        return {
            "rag_results": (
                "The knowledge base has no information on this. Please contact Food Factory at "
                "+91-80-4567-8900 or reservations@foodfactory.in."
            ),
            "sources": [],
        }

    context = "\n\n---\n\n".join(doc.page_content for doc, _ in hits)
    response = llm.invoke([
        SystemMessage(content=(
            "You are a helpful assistant for Food Factory, a fine-dining restaurant in Bangalore. "
            "Answer the customer's question using ONLY the knowledge-base excerpts provided. Include specific "
            "prices, times, capacities or requirements when relevant, and booking contacts when useful. "
            "If the excerpts do not fully answer the question, say what is missing."
        )),
        HumanMessage(content=f"Knowledge-base excerpts:\n{context}\n\nCustomer question: {query}"),
    ])
    print(f"✅ RAG: answered from {len(hits)} chunks")
    return {"rag_results": response.content, "sources": [chunk_label(doc.page_content) for doc, _ in hits]}


update = restaurant_rag_agent({"query": "Provide an overview of Food Factory"})
print("Sources:", update["sources"], "\n")
print(update["rag_results"])

---
## 🌐 Part 4: The Web Research Agent

For questions about current, changing information the web agent searches DuckDuckGo through the `ddgs`
package (no API key), keeps the top `WEB_RESULTS` hits, and asks the model to extract the relevant facts
**with the URLs they came from**. Today's date goes into the prompt so "latest" and "current" are read
correctly. A failed search (no network, rate limit) is reported in the result instead of crashing the graph.

In [ ]:
# ============================================================================
# WEB RESEARCH AGENT, TESTED ON ITS OWN
# ============================================================================
WEB_RESULTS = 5  # search hits passed to the model; keep it small to limit tokens


def web_search_agent(state: AgentState) -> dict:
    query = state.get("standalone_query") or state["query"]
    try:
        results = DDGS().text(query, max_results=WEB_RESULTS)
    except Exception as e:  # network errors, rate limits
        print(f"❌ Web search failed: {e}")
        return {"web_results": f"Web search failed: {e}", "sources": []}

    formatted = "\n\n".join(f"[{i}] {r['title']}\nURL: {r['href']}\n{r['body']}" for i, r in enumerate(results, 1))
    response = llm.invoke([
        SystemMessage(content=(
            f"You are a web research analyst. Today is {date.today():%d %B %Y}. From the search results, extract "
            "the information relevant to the query: 3-5 key findings with important details, each followed by "
            "the URL it came from. Use only the search results; say so if they do not answer the query."
        )),
        HumanMessage(content=f"Query: {query}\n\nSearch results:\n{formatted}"),
    ])
    print(f"✅ Web research: {len(results)} results")
    return {"web_results": response.content, "sources": [r["href"] for r in results]}


update = web_search_agent({"query": "What is the latest release of the LangGraph Python library?"})
print("Sources:", *update["sources"], sep="\n  ")
print()
print(update["web_results"])

**Reading the output:** the sources are the URLs the search actually returned, taken from the results in code.
Compare them with the URLs the model cites in its findings: every cited URL should be one of them.

---
## 🤖 Part 5: The LLM Agent

Some questions need neither the knowledge base nor the web: "write a function", "explain a concept". The LLM
agent answers those from the model's own knowledge.

In [ ]:
# ============================================================================
# LLM AGENT
# ============================================================================
def llm_agent(state: AgentState) -> dict:
    query = state.get("standalone_query") or state["query"]
    response = llm.invoke([
        SystemMessage(content=(
            "You are an expert assistant for technology, data engineering and general knowledge. "
            "Give an accurate, well-structured answer, with a short example where it helps."
        )),
        HumanMessage(content=query),
    ])
    print(f"✅ LLM agent: {len(response.content)} characters")
    return {"llm_response": response.content, "sources": []}


print("✅ llm_agent defined")

---
## 🔀 Part 6: The Router

Asking the model to "respond with ONLY the route name" and falling back to keyword lists when the reply is
anything else is fragile. With **structured output** the reply *is* a `RouteDecision` object whose `route`
field can only be one of three values, so there is nothing to parse and no fallback to maintain.

The router also sees the last few questions on the thread and writes a **standalone** version of the new one:
"what about 40 people?" becomes a full question about Food Factory, which both the routing decision and the
specialist's search need.

The routing *function* (`route_query`) makes no model call: it only reads `route_decision` from the state.

The same typed router, dispatching to specialists that are graph nodes, is Part 1 of
[`05_meta_controller.ipynb`](../../m26_collaboration_parallel_agents/optional/04a_meta_controller.ipynb)
(module 08). That notebook then goes one level up: a meta-controller that picks a whole architecture instead of
one specialist.

In [ ]:
# ============================================================================
# ROUTER NODE AND PURE ROUTING FUNCTION
# ============================================================================
class RouteDecision(BaseModel):
    standalone_query: str = Field(
        description="The user's question rewritten to be fully understandable without the conversation history "
        "(unchanged if it already is)"
    )
    route: Literal["web_research", "rag", "llm"] = Field(
        description="web_research: current, time-sensitive or live information (prices, news, releases); "
        "rag: anything about the Food Factory restaurant; "
        "llm: general knowledge, reasoning, writing or code that needs no external data"
    )
    reason: str = Field(description="One short sentence explaining the choice")


router_llm = llm.with_structured_output(RouteDecision)


def router_agent(state: AgentState) -> dict:
    recent = state.get("history", [])[-3:]
    history_text = "\n".join(f"Q: {h['query']}\nA: {h['answer'][:300]}" for h in recent) or "(none)"
    decision = router_llm.invoke([
        SystemMessage(content=f"You route user questions to the right specialist. Today is {date.today():%d %B %Y}."),
        HumanMessage(content=f"Conversation so far:\n{history_text}\n\nNew question: {state['query']}"),
    ])
    print(f"🔀 Router: {decision.route} ({decision.reason})")
    return {
        "standalone_query": decision.standalone_query,
        "route_decision": decision.route,
        "route_reason": decision.reason,
        # reset this question's results so nothing from an earlier question on the thread leaks into the summary
        "web_results": "", "rag_results": "", "llm_response": "", "sources": [],
    }


def route_query(state: AgentState) -> Literal["web_research", "rag", "llm"]:
    """Pure: reads the router's decision from the state."""
    return state["route_decision"]


print("✅ Router defined. Possible routes:", RouteDecision.model_fields["route"].annotation.__args__)

---
## 📝 Part 7: The Summarization Agent

The summarizer takes the result of **this question's** route only, and writes the final answer. It is told
not to add facts of its own, so the answer stays as grounded as the specialist's result. It also appends the
question and answer to `history`.

In [ ]:
# ============================================================================
# SUMMARIZATION AGENT
# ============================================================================
RESULT_KEY = {"web_research": "web_results", "rag": "rag_results", "llm": "llm_response"}


def summarization_agent(state: AgentState) -> dict:
    route = state["route_decision"]
    result = state[RESULT_KEY[route]]
    sources = "\n".join(f"- {s}" for s in state.get("sources", [])) or "(none)"
    response = llm.invoke([
        SystemMessage(content=(
            "You write the final answer for the user from a specialist's result. Answer the question directly, "
            "use clear markdown sections where they help, and end with brief key takeaways. Use only the "
            "information in the specialist's result: do not add facts. Reply with the answer only."
        )),
        HumanMessage(content=(
            f"Question: {state['standalone_query']}\nRoute: {route}\n\n"
            f"Specialist result:\n{result}\n\nSources:\n{sources}"
        )),
    ])
    print(f"✅ Summary: {len(response.content)} characters")
    return {
        "final_summary": response.content,
        "history": [{"query": state["standalone_query"], "answer": response.content}],
    }


print("✅ summarization_agent defined")

---
## 🏗️ Part 8: Assembling the Graph

`START → router → (web_research | rag | llm) → summarizer → END`, compiled with a `InMemorySaver` checkpointer so
each `thread_id` keeps its own `history`.

In [ ]:
# ============================================================================
# BUILD AND COMPILE THE GRAPH
# ============================================================================
workflow = StateGraph(AgentState)
workflow.add_node("router", router_agent)
workflow.add_node("web_research", web_search_agent)
workflow.add_node("rag", restaurant_rag_agent)
workflow.add_node("llm", llm_agent)
workflow.add_node("summarizer", summarization_agent)

workflow.add_edge(START, "router")
workflow.add_conditional_edges("router", route_query, ["web_research", "rag", "llm"])
for specialist in ("web_research", "rag", "llm"):
    workflow.add_edge(specialist, "summarizer")
workflow.add_edge("summarizer", END)

research_agent = workflow.compile(checkpointer=InMemorySaver())

print("✅ Graph compiled. Edges:")
for edge in research_agent.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### 8.1 ▶️ A helper to ask one question

`ask()` streams the run with `stream_mode="updates"`, which yields one update per node **as it finishes**, so
the list of node names is the actual path the question took. It then reads the final state from the
checkpointer and shows the answer.

In [ ]:
# ============================================================================
# ASK HELPER
# ============================================================================
def ask(graph, question: str, thread_id: str, show_answer: bool = True) -> tuple[dict, list[str]]:
    config = {"configurable": {"thread_id": thread_id}}
    nodes_run = []
    for update in graph.stream({"query": question}, config, stream_mode="updates"):
        nodes_run.extend(update.keys())
    state = graph.get_state(config).values

    print(f"\n👤 [{thread_id}] {question}")
    if state["standalone_query"] != question:
        print(f"📝 Rewritten as: {state['standalone_query']}")
    print(f"🧭 Path: {' → '.join(nodes_run)}")
    print(f"📚 Sources: {state['sources'] or '(none)'}")
    if show_answer:
        display(Markdown(state["final_summary"]))
    return state, nodes_run


print("✅ ask() defined")

---
## 🎬 Part 9: Three Questions, Three Routes

One question for each route, each on its own thread. After the runs, code compares the path each question
actually took with the path we expected.

In [ ]:
# ============================================================================
# ONE QUESTION PER ROUTE, PATHS CHECKED IN CODE
# ============================================================================
questions = [
    ("What is Nvidia's current P/E ratio?", "web_research", "web-1"),
    ("Write a Python function to calculate Fibonacci numbers", "llm", "code-1"),
    ("Can I host a private event for 20 people at Food Factory?", "rag", "food-1"),
]

runs = []
for text, expected, thread in questions:
    state, path = ask(research_agent, text, thread)
    runs.append((text, expected, state["route_decision"], path))
    print("=" * 80)

print("\n--- Route check ---")
for text, expected, got, path in runs:
    ok = got == expected and path == ["router", expected, "summarizer"]
    print(f"{'✅' if ok else '❌'} expected {expected:<12} got {got:<12} path {' → '.join(path)} | {text[:45]}")

**Reading the output:** a ✅ means the router chose the expected specialist *and* exactly that specialist ran,
between the router and the summarizer. For the web question, look at the sources: they are today's search
results, so the figure you see changes over time. For the restaurant question the answer should be built from
the private dining room's 8-24 guest capacity retrieved in Part 1.

---
## 💬 Part 10: Follow-up Questions and Memory

Now a follow-up, "What about 40 people?", asked twice:
- on thread `food-1`, which already holds the private-event question, and
- on a new thread `food-2`, which has no history.

The router sees the thread's `history`, so on `food-1` it can rewrite the follow-up into a full question about
Food Factory. On `food-2` there is nothing to rewrite it from.

In [ ]:
# ============================================================================
# THE SAME FOLLOW-UP ON TWO THREADS
# ============================================================================
followup = "What about 40 people?"
before = {t: len(research_agent.get_state({"configurable": {"thread_id": t}}).values.get("history", []))
          for t in ("food-1", "food-2")}

state_1, path_1 = ask(research_agent, followup, "food-1")
print("=" * 80)
state_2, path_2 = ask(research_agent, followup, "food-2")

print("\n--- Memory check ---")
print(f"{'✅' if before == {'food-1': 1, 'food-2': 0} else '❌'} history before the follow-up: {before}")
print(f"{'✅' if len(state_1['history']) == 2 else '❌'} food-1 now holds {len(state_1['history'])} answered questions")
print(f"{'✅' if state_1['route_decision'] == 'rag' else '❌'} food-1 follow-up routed to rag: {state_1['route_decision']}")
print(f"   food-1 rewrite: {state_1['standalone_query']}")
print(f"   food-2 rewrite: {state_2['standalone_query']} (route: {state_2['route_decision']})")

**Reading the output:** on `food-1` the rewrite should name Food Factory and a private event, and the answer
comes from the knowledge base (the private dining room seats at most 24, so look for what it offers 40 guests
instead). On `food-2` the same five words carry no context: compare its rewrite and route with `food-1`'s.

To try your own questions, call `ask(research_agent, "your question", "any-thread-id")` in a new cell.

---
## 📝 Summary

In this notebook, we learned:

### 1. Specialist agents
- The **RAG agent** answers from retrieved chunks only, and skips the model call when nothing relevant is found.
- The **web agent** searches DuckDuckGo with `ddgs` and keeps the result URLs as its sources.
- The **LLM agent** handles questions that need no external data.

### 2. Typed, pure routing
- `with_structured_output(RouteDecision)` returns a `Literal` route: no string parsing, no keyword fallback.
- `route_query` only reads `route_decision`, and `stream_mode="updates"` shows the path each question took.

### 3. Summarization and memory
- The summarizer uses only this question's result; the router resets per-question fields on every turn.
- A `history` reducer and a checkpointer let the router turn a follow-up into a standalone question.

`langgraph_research_agent_single.py` in this folder is the same router → specialist → summarizer design as one
command-line script (Tavily search and a small ML knowledge base).

### Next Steps
- **`14_multi_agent_research_system.ipynb`**: a one-shot planner fans out parallel search agents with `Send`, and a
  quality loop sends the report back for revision